In [ ]:
import os, glob, re
import pandas as pd

BASE_DIR = 'input/Density_Scenario_Paper'  # <- 修改路径

tif_files = sorted(glob.glob(os.path.join(BASE_DIR, '**/*.tif'), recursive=True))
print(f'Total tif files: {len(tif_files)}')
print()

# Print ALL tif basenames
for f in tif_files:
    print(os.path.relpath(f, BASE_DIR))

In [ ]:
# Parse filename structure
# Example: Guinea_Ecuatorial_low80_highdensity_2050.tif
# Pattern: {country_name}_{something}_{scenario}_{year}.tif

records = []
for f in tif_files:
    name = os.path.splitext(os.path.basename(f))[0]
    folder = os.path.basename(os.path.dirname(f))
    
    # Extract year
    year_m = re.search(r'(\d{4})$', name)
    year = year_m.group(1) if year_m else None
    
    # Extract scenario keyword
    scenario = None
    for kw in ['highdensity', 'high_density', 'consdensity', 'cons_density', 
                'consolidated', 'lowdensity', 'low_density',
                'highdensit', 'consdensit', 'lowdensit']:
        if kw.lower() in name.lower():
            scenario = kw
            break
    
    # Extract any numeric part (like low80, low60...)
    num_m = re.search(r'(low|high|cons)(\d+)', name, re.IGNORECASE)
    num_suffix = num_m.group(0) if num_m else None
    
    records.append({
        'folder': folder,
        'basename': name,
        'year': year,
        'scenario': scenario,
        'num_suffix': num_suffix,
        'size_mb': os.path.getsize(f) / 1024 / 1024
    })

df = pd.DataFrame(records)
print('=== Scenario keyword counts ===')
print(df['scenario'].value_counts().to_string())
print()
print('=== Year counts ===')
print(df['year'].value_counts().to_string())
print()
print('=== Numeric suffix counts ===')
print(df['num_suffix'].value_counts().to_string())
print()
print('=== By folder ===')
print(df.groupby('folder')['scenario'].value_counts().to_string())

In [ ]:
# Show all unique filename patterns (replace country name with placeholder)
# to find the template
print('=== Full basename sample (first 30) ===')
for r in records[:30]:
    print(f"  [{r['folder']}]  {r['basename']}  →  scenario={r['scenario']}  year={r['year']}")

In [ ]:
# How many tifs per country folder?
# If 3 scenarios × years → each country should have 3 (or 6 if 2015+2050)
print('=== Files per folder ===')
folder_counts = df.groupby('folder').size().sort_values(ascending=False)
print(folder_counts.to_string())
print()

# Cross-tab: folder × scenario
print('=== Folder × Scenario cross-tab ===')
ct = pd.crosstab(df['folder'], df['scenario'])
print(ct.to_string())

In [ ]:
# Read one tif with rasterio to understand its content
try:
    import rasterio
    from rasterio.plot import show
    import matplotlib.pyplot as plt
    import numpy as np

    # Pick one file per scenario type
    for scenario_kw in ['highdensity', 'consdensity', 'lowdensity', 
                         'highdensit', 'consdensit', 'lowdensit']:
        matches = [f for f in tif_files if scenario_kw.lower() in f.lower()]
        if matches:
            sample = matches[0]
            with rasterio.open(sample) as src:
                meta = src.meta
                bounds = src.bounds
                data = src.read(1)
                nodata = src.nodata
                valid = data[data != nodata] if nodata is not None else data.flatten()
                unique_vals = np.unique(valid)
            print(f"--- Scenario: {scenario_kw} ---")
            print(f"File: {os.path.basename(sample)}")
            print(f"CRS: {meta['crs']}")
            print(f"Shape: {meta['height']} × {meta['width']}")
            print(f"Bounds: {bounds}")
            print(f"Dtype: {meta['dtype']}")
            print(f"NoData: {nodata}")
            print(f"Unique values ({len(unique_vals)} total): {unique_vals[:20]}")
            print()
            break  # just one for now

except ImportError:
    print('rasterio not installed — run: pip install rasterio')
    print('Checking with gdal instead...')
    import subprocess
    sample = tif_files[0]
    result = subprocess.run(['gdalinfo', sample], capture_output=True, text=True)
    print(result.stdout[:2000])

In [ ]:
# Compare HIGH vs CONS vs LOW for the same country/city
# Find a country that has all 3 scenarios
import rasterio
import numpy as np

scenario_map = {}
for f in tif_files:
    name = os.path.basename(f).lower()
    folder = os.path.basename(os.path.dirname(f))
    for kw in ['highdensity','highdensit','high_density']:
        if kw in name: scenario_map.setdefault(folder, {})['high'] = f
    for kw in ['consdensity','consdensit','cons_density','consolidated']:
        if kw in name: scenario_map.setdefault(folder, {})['cons'] = f
    for kw in ['lowdensity','lowdensit','low_density']:
        if kw in name: scenario_map.setdefault(folder, {})['low'] = f

# Find folder with all 3
complete = {k: v for k, v in scenario_map.items() if len(v) == 3}
print(f'Folders with all 3 scenarios: {list(complete.keys())}')

if complete:
    folder_name = list(complete.keys())[0]
    scenarios = complete[folder_name]
    print(f'\nComparing scenarios for: {folder_name}')
    
    for sc, fpath in scenarios.items():
        with rasterio.open(fpath) as src:
            data = src.read(1)
            nodata = src.nodata
            valid = data[data != nodata] if nodata is not None else data.flatten()
            valid = valid[valid > 0]  # remove zeros too
        print(f"  {sc}: {os.path.basename(fpath)}")
        print(f"       pixels_with_data={len(valid)}, sum={valid.sum():.0f}, mean={valid.mean():.2f}")

In [ ]:
# Visualize: side-by-side map of high vs cons vs low for one country
import rasterio
import matplotlib.pyplot as plt
import numpy as np

if complete:
    folder_name = list(complete.keys())[0]
    scenarios = complete[folder_name]
    
    fig, axes = plt.subplots(1, 3, figsize=(18, 6))
    
    for ax, (sc, fpath) in zip(axes, scenarios.items()):
        with rasterio.open(fpath) as src:
            data = src.read(1).astype(float)
            nodata = src.nodata
            if nodata is not None:
                data[data == nodata] = np.nan
            bounds = src.bounds
        
        im = ax.imshow(data, cmap='YlOrRd', aspect='auto',
                       extent=[bounds.left, bounds.right, bounds.bottom, bounds.top])
        ax.set_title(f'{sc.upper()} density\n{os.path.basename(fpath)[:40]}', fontsize=9)
        plt.colorbar(im, ax=ax, shrink=0.7)
    
    fig.suptitle(f'Three density scenarios: {folder_name}', fontsize=12)
    plt.tight_layout()
    plt.savefig('scenario_comparison.png', dpi=100, bbox_inches='tight')
    plt.show()
    print('Saved: scenario_comparison.png')

In [ ]:
# KEY QUESTION: what do pixel values represent?
# Is it 0/1 (binary: urban/non-urban)?
# Or a continuous density value?
# Or population count per pixel?

if complete:
    folder_name = list(complete.keys())[0]
    f_high = complete[folder_name]['high']
    
    with rasterio.open(f_high) as src:
        data = src.read(1)
        nodata = src.nodata
        res = src.res  # pixel size in degrees
        print(f'Pixel size: {res[0]:.6f} × {res[1]:.6f} degrees')
        print(f'≈ {res[0]*111:.2f} × {res[1]*111:.2f} km at equator')
    
    valid = data[data != nodata] if nodata is not None else data.flatten()
    print(f'\nValue distribution:')
    unique, counts = np.unique(valid, return_counts=True)
    print(f'  Unique values: {unique[:30]}')
    print(f'  Counts:        {counts[:30]}')
    print(f'  Total unique: {len(unique)}')
    
    if len(unique) <= 5:
        print('\n→ Looks like BINARY or CATEGORICAL raster (urban mask)')
    else:
        print('\n→ Looks like CONTINUOUS raster (density or population)')

In [ ]:
# CRITICAL: relationship between tif and shapefile
# The shapefile has polygon boundaries for each city
# The tif might be the RASTER version of the urban extent
# OR the tif might give additional info INSIDE the polygon

# Let's check: does the tif cover the whole country or just city footprints?
if tif_files:
    # Pick Guinea Ecuatorial (the example you gave)
    geq_tifs = [f for f in tif_files if 'Guinea' in f or 'GEQ' in f or 'Ecuatorial' in f]
    if geq_tifs:
        print('Guinea Ecuatorial tifs found:')
        for f in geq_tifs:
            print(f'  {os.path.basename(f)}')
        
        with rasterio.open(geq_tifs[0]) as src:
            print(f'\nBounds: {src.bounds}')
            print(f'CRS: {src.crs}')
    else:
        print('Guinea Ecuatorial tifs not found with that name — checking folder:')
        geq_tifs = [f for f in tif_files if 'GEQ' in os.path.dirname(f) or 
                    'Insular' in f or 'CA' in os.path.dirname(f)]
        for f in geq_tifs[:5]:
            print(f'  {os.path.relpath(f, BASE_DIR)}')